## Define baby's identification data

In [10]:
from pydantic import ValidationError

from utils.files import read_json
from models.baby import Baby

baby_data: dict = read_json("data/babies.json")
baby_list: list[Baby] = []
try:
    for baby_entry in baby_data:
        baby_data = Baby.model_validate(baby_entry)
        baby_list.append(baby_data)
    print("Your babies list length is:", len(baby_list))
except ValidationError as e:
    print("Validation error:", e)

Your babies list length is: 1


### Select desired baby to add new data

In [12]:
baby: Baby = None  # Default to the first baby if only one exists
baby_growth_file = None

if len(baby_list) > 1:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    baby_selection = widgets.Dropdown(
        options=[(f"{b.name} {b.last_name}", b) for b in baby_list],
        value=baby_list[0],
        description="Select Baby:",
        layout=widgets.Layout(width="350px"),
    )
    button = widgets.Button(description="Confirm choice",
                            button_style="primary")
    output = widgets.Output()

    def on_button_click(_):
        """Handle button click event."""
        global baby, baby_growth_file
        with output:
            baby = baby_selection.value
            baby_growth_file = f"{baby.id}.csv"
            clear_output(wait=True)
            print("You've selected:", baby.name, baby.last_name)

    button.on_click(on_button_click)
    display(widgets.VBox([baby_selection, button, output]))

elif len(baby_list) == 1:
    baby = baby_list[0]
    baby_growth_file = f"{baby.id}.csv"
    print(
        f"Only one baby found: {baby.name} {baby.last_name}. Using this baby.")
else:
    raise ValueError("No babies found in the data.")

Only one baby found: Agatha Sánchez López. Using this baby.


In [ ]:
import polars as pl
from datetime import datetime

from utils.files import csv_to_dataframe
from models.registry import GrowthRegistry

GROWTH_SCHEMA = {
    "weight_kg": pl.Float64,
    "height_cm": pl.Float64,
    "head_circum_cm": pl.Float64,
    "date": pl.String,
}

growth_data_df = csv_to_dataframe(baby_growth_file, schema=GROWTH_SCHEMA)

if growth_data_df.is_empty():
    print("No growth data found. Registering first growth data...")
    first_growth_registry = GrowthRegistry(
        weight_kg=baby.birth_weight_kg,
        height_cm=baby.birth_height_cm,
        head_circum_cm=baby.birth_head_circum_cm,
        date=baby.birth_datetime.split(" ")[0],
    )
    growth_data_df = pl.concat(
        [growth_data_df,
         pl.DataFrame([first_growth_registry.model_dump()])])
    growth_data_df.write_csv(baby_growth_file)
    print("First growth registry registered:", growth_data_df.head())
else:
    print("Growth data found:", growth_data_df.head())
    wants_to_register_new_data = True
    while wants_to_register_new_data:
        print("Do you want to register new growth data? (yes/no)")
        user_input = input().lower()
        if user_input == "yes":
            set_weight_kg = False
            is_height_set = False
            is_head_circum_set = False
            print("Please enter the new growth data:")
            date = input("Date (YYYY-MM-DD): ")
				# TODO -> implement date validation
            if not date:
                date = datetime.today().strftime("%Y-%m-%d")
                print(f"No date provided. Using today's date: {date}")
            # while not set_weight_kg:
            #     weight_kg = float(input("Weight (kg): "))
            #     if weight_kg <= 0 or not isinstance(
            #             weight_kg, (int, float)) or weight_kg is None:
            #         print(
            #             "Weight is a mandatory field and must be a positive number. Please try again."
            #         )
            #     else:
            #         set_weight_kg = True
            # while not is_height_set:
            #     height_cm = float(input("Height (cm): "))
				# 	 if head_circum_cm < 0 or not isinstance(weight_kg, (int, float)):
				# 		print("Height must be a positive number")
				# 		continue
				# 	 elif head_circum_cm is None:
				# 		height_none_choice = True

            # head_circum_cm = float(input("Head circumference (cm): "))
        elif user_input == "no":
            wants_to_register_new_data = False
        else:
            print("Invalid input. Please enter 'yes' or 'no'.")

FileNotFoundError: No such file or directory (os error 2): data/growth.csv